In [1]:
from typing import Annotated , Literal , TypedDict  
from langgraph.types import Command
from langchain_tavily import TavilySearch
from langgraph.graph import MessagesState, END , StateGraph, START ,END
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_groq import ChatGroq

In [2]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [3]:
Tavily_api_key = os.getenv("Tavily_api_key")
search_tool = TavilySearch(
    max_results=5,
    topic="general")


In [4]:
members = ['Researcher', 'Reporter', 'FINISH']

In [5]:
LLM = ChatGroq(model = "llama-3.3-70b-versatile")

In [6]:
class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH."""
    next: Literal['Researcher', 'Reporter', 'FINISH']

In [7]:
class State(MessagesState):
    next:str

In [8]:
researcher_prompt = """You are a Senior Materials Scientist and Structural Engineer. Your role is to act as a data-mining engine for an AI decision-support tool.Your Task: When provided with a construction requirement (e.g., 'load-bearing wall for a 3-story building'), you must identify three potential eco-friendly materials and provide a data-driven technical profile for each.Data Points Required:Structural Specs: Compressive/Tensile strength ($MPa$), density ($kg/m^3$), and thermal conductivity ($\lambda$).Sustainability Metrics: Embodied carbon ($kgCO_2e/kg$), water footprint, and recyclability percentage.Economic Data: Estimated cost per unit and regional availability.Standard Comparison: Always include a baseline comparison against a traditional material (e.g., OPC Concrete or Red Brick).Constraint: Maintain a strictly objective, technical tone. Use LaTeX for all units and chemical formulas. Present your findings in a structured Markdown table for the Report Agent to process."""

In [9]:
research_agent = create_agent(
    LLM,
    tools = [search_tool],
    system_prompt=researcher_prompt
) 

In [10]:
report_prompt = """You are a Lead Sustainability Consultant and Architectural Advisor. Your role is to take raw technical data from the Research Agent and transform it into a 'Sustainable Material Recommendation Report' for a project developer.

Your Task: Evaluate the research findings and provide a strategic recommendation based on three pillars: Structural Safety, Environmental ROI, and Cost-Effectiveness.

Report Structure:

Executive Recommendation: Which material is the 'Best Fit' and why?

The 'Green Premium' Analysis: Compare the upfront cost vs. the long-term environmental/operational savings (e.g., energy efficiency).

Compliance & Certification: Mention how these materials contribute to LEED, BREEAM, or local green building codes.

Implementation Risks: Identify potential bottlenecks, such as specialized labor requirements or supply chain lead times.

Constraint: Use professional, persuasive language. Ensure the report is scannable with clear headings and bolded key takeaways. Your goal is to help a human make a final decision and Your  task is to take the technical research provided and format it into a professional Markdown-style Construction Material Report."""

In [11]:
reporter_agent = create_agent(
    LLM,
    tools = [],
    system_prompt=report_prompt
) 

In [12]:
system_prompt=f"""
You are a supervisor, tasked with managing a conversation between the following workers: {members}. 
Given the following user request, respond with the worker to act next. 
Each worker will perform a task and respond with their results and status. 
When finished, respond with FINISH.
"""

In [13]:
def supervisor_node(state: State) -> Command[Literal["Researcher", "Reporter", "__end__"]]:
    
    messages = [{"role": "system", "content": system_prompt},] + state["messages"]
    
    response = LLM.with_structured_output(Router).invoke(messages)
    
    goto = response["next"]
    
    print("Supervisor initialized")
    
    print(goto)
    
    if goto == "FINISH":
        goto = END
        
    return Command(goto=goto, update={"next": goto})

In [14]:
def Research_node(state: State) -> Command[Literal["supervisor"]]:
    
    result = research_agent.invoke(state)
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="Researcher")
            ]
        },
        goto="supervisor",
    )

In [15]:
def Reporter_node(state: State) -> Command[Literal["supervisor"]]:
    
    result = reporter_agent.invoke(state)
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="Reporter")
            ]
        },
        goto="supervisor",
    )

In [16]:
graph=StateGraph(State)
graph.add_node("supervisor",supervisor_node)
graph.add_node("Researcher", Research_node)
graph.add_node("Reporter", Reporter_node)

In [17]:
graph.add_edge(START,"supervisor")


In [18]:
app = graph.compile()

In [19]:
# for s in app.stream({"messages": [("user", "I am designing a 3-story residential building in a tropical climate. I need a load-bearing wall material that is sustainable, has a high thermal mass to reduce cooling costs, and fits a moderate budget. What do you recommend?")]}, subgraphs=True):
#     print(s)
#     print("----")

In [20]:
input = {"messages": [("user", "I am designing a 3-story residential building in a tropical climate. I need a load-bearing wall material that is sustainable, has a high thermal mass to reduce cooling costs, and fits a moderate budget. What do you recommend?")]}

In [21]:
final_state = app.invoke(input)


Supervisor initialized
Researcher
Supervisor initialized
Reporter
Supervisor initialized
FINISH


In [22]:
report_content = ""
for message in reversed(final_state["messages"]):
    # Check if the message came from the Reporter node
    if getattr(message, 'name', None) == "Reporter":
        report_content = message.content
        break

# Save to a file
if report_content:
    with open("Material_Recommendation_Report.md", "w") as f:
        f.write(report_content)
    print("Report successfully saved to Material_Recommendation_Report.md")
else:
    print("No report content found.")

Report successfully saved to Material_Recommendation_Report.md
